In [98]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import Parser.parser as parser
from AST.ast_tree import ASTTransformer
from Semantic.semantic_analyzer import semantic_analysis, SemanticError
from Quadruples.quadruple_generator import generate_quadruples, print_quadruples
from Quadruples.interpreter import interpret
from lark import exceptions as lark_exceptions

import importlib, Parser.parser as parser, AST.ast_tree as ast_tree, Semantic.semantic_analyzer as semantic_analyzer
importlib.reload(parser)
importlib.reload(ast_tree)
importlib.reload(semantic_analyzer)

<module 'semantic_analyzer' from '/Users/debbelzz/TEC/Compi/Compiler/src/semantic_analyzer.py'>

In [100]:
%cd ..

/


In [ ]:
import os
import pprint
from pathlib import Path

from Semantic.semantic_analyzer import SemanticError

code = """
  program main {
  begin;
    write("Hello, World!");
  end;
}
"""

try:
    parse_tree = parser.parse_code(code)
except lark_exceptions.UnexpectedInput as e:
    line = e.line
    column = e.column
    expected = ", ".join(sorted(e.expected)) if hasattr(e, "expected") else "unknown"
    print(f"[SyntaxError] line {line}, column {column}: unexpected token.")
    print(f"Expected: {expected}")
    print(e.get_context(code, span=40))

try:
    parse_tree = parser.parse_code(code)
    ast = ASTTransformer().transform(parse_tree)
    symbol_table = semantic_analysis(ast)
    print("OK", symbol_table)
except lark_exceptions.UnexpectedInput as e:
    print(f"[SyntaxError] line {e.line}, col {e.column}")
    print(e.get_context(code, span=40))
except lark_exceptions.VisitError as e:
    print(f"[ASTError] rule={e.rule}")
    print(f"Cause: {type(e.orig_exc).__name__}: {e.orig_exc}")
except SemanticError as e:
    print(f"[SemanticError] {e}")

quadruples = generate_quadruples(ast)
# print_quadruples(quadruples)
interpret(quadruples)

{}
OK {}
Hello, World!


In [97]:
print(parse_tree.pretty())

start
  var_declaration
    i
    n
    x
    type	int
  begin
    write
      constant	"factorial for"
    assignment
      x
      constant	1
    assignment
      n
      constant	5
    for_stmt
      assignment
        i
        constant	1
      relative_expression
        variable	i
        rel_op	<
        variable	n
      variable
        i
        step_operator	++
      block
        assignment
          x
          multiplicative_expression
            variable	x
            mul_op	*
            variable	i
    write
      variable	x

